In [ ]:
!pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 105.3 MB/s eta 0:00:00


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, glob, time, json
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from monai.networks import nets
from monai.transforms import (Compose, RandFlip, RandRotate, RandScaleIntensity,
                              RandShiftIntensity, RandGaussianNoise)

# ---- device / AMP (bỏ qua nếu notebook đã định nghĩa) ----
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True
_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if _bf16 else torch.float16
SEED = 42

def to_device_normalize(x, y):
    x = x.to(device, non_blocking=True).float().div_(255.0)   # uint8 -> [0,1] trên GPU
    y = y.to(device, non_blocking=True)
    return x, y

# ---- augmentation 3D (chỉ dùng cho Training) ----
# Bảo thủ cho OCT: lật in-plane, xoay nhẹ, biến đổi cường độ + nhiễu nhẹ.
def make_train_transform():
    return Compose([
        RandFlip(prob=0.5, spatial_axis=1),                         # lật H
        RandFlip(prob=0.5, spatial_axis=2),                         # lật W
        RandRotate(range_x=0.10, range_y=0.10, range_z=0.10,        # ±~6°
                   prob=0.5, mode="bilinear", padding_mode="zeros", keep_size=True),
        RandScaleIntensity(factors=0.10, prob=0.5),                 # *(1±0.1)
        RandShiftIntensity(offsets=10.0, prob=0.5),                 # ±10 trên thang [0,255]
        RandGaussianNoise(prob=0.3, std=5.0),
    ])


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [ ]:
import os, time, shutil, subprocess, json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ---- paths / knobs (match your notebook's CONFIG cell) ----
DRIVE_ARCHIVE = "/content/drive/MyDrive/MasterBKDN/Thesis/glaucoma_all_96.tar.zst"
LOCAL_ARCHIVE = "/content/glaucoma_all_96.tar.zst"
LOCAL_DATA_DIR = "/content/dataset"          # extracted subset dir goes here
SUBSET_DIRNAME = "glaucoma_all_96"         # top-level folder inside the archive

SPLITS = ("Training", "Validation", "Test")
BATCH_SIZE  = 18
SEED        = 42
CACHE_IN_RAM = True                          # keep matching your notebook
NUM_WORKERS  = max(2, os.cpu_count() or 2)
device = "cuda" if torch.cuda.is_available() else "cpu"


def ensure_extracted():
    """Copy the archive Drive->local SSD (once) and extract it (once)."""
    subset_dir = os.path.join(LOCAL_DATA_DIR, SUBSET_DIRNAME)
    if os.path.isdir(subset_dir) and any(os.scandir(subset_dir)):
        print(f"[data] already extracted at {subset_dir}")
        return subset_dir

    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    if not os.path.exists(LOCAL_ARCHIVE):
        print(f"[data] copying archive Drive->local SSD ...")
        t0 = time.time()
        shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
        print(f"[data] copied in {time.time()-t0:.1f}s")

    # Colab base images may lack the zstd CLI -- install it once. GNU tar there
    # supports `tar -I zstd`; we also try `tar --zstd` and a python fallback.
    print("[data] extracting ...")
    t0 = time.time()
    try:
        subprocess.run("apt-get -qq install -y zstd",
                       shell=True, check=False)
    except Exception:
        pass
    extracted = False
    for cmd in (f"tar -I zstd -xf {LOCAL_ARCHIVE} -C {LOCAL_DATA_DIR}",
                f"tar --zstd -xf {LOCAL_ARCHIVE} -C {LOCAL_DATA_DIR}",
                f"zstd -dc {LOCAL_ARCHIVE} | tar -xf - -C {LOCAL_DATA_DIR}"):
        if subprocess.run(cmd, shell=True).returncode == 0:
            extracted = True
            break
    if not extracted:
        # Pure-python fallback (no system zstd needed): pip install zstandard.
        subprocess.run("pip -q install zstandard", shell=True, check=False)
        import zstandard, tarfile, io
        dctx = zstandard.ZstdDecompressor()
        with open(LOCAL_ARCHIVE, "rb") as fh, dctx.stream_reader(fh) as r:
            with tarfile.open(fileobj=r, mode="r|") as tf:
                tf.extractall(LOCAL_DATA_DIR)
    print(f"[data] extracted in {time.time()-t0:.1f}s")
    return subset_dir


class OCTMemmapDataset(Dataset):
    """
    Map-style dataset over the consolidated per-split arrays.

    volumes .npy is (N,1,200,200,200) uint8 (channel axis pre-baked), labels .npy
    is (N,) int64. We open the volumes with mmap_mode='r' so random access is a
    cheap page read -- works great with DataLoader(shuffle=True). With
    CACHE_IN_RAM we read everything into RAM once (uint8 stays uint8).
    """
    def __init__(self, subset_dir, split, cache_in_ram=False):
        self.vol_path = os.path.join(subset_dir, f"{split}_volumes.npy")
        self.lbl_path = os.path.join(subset_dir, f"{split}_labels.npy")
        self.labels = np.load(self.lbl_path)               # tiny (N,) int64
        if cache_in_ram:
            self.volumes = np.load(self.vol_path)          # full read into RAM
        else:
            # Lazy mmap. (With num_workers>0, prefer opening per-worker via
            # worker_init_fn; the notebook forces workers=0 when CACHE_IN_RAM.)
            self.volumes = np.load(self.vol_path, mmap_mode="r")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # .copy() detaches from the memmap -> writable, worker/pin_memory-safe,
        # and never aliases the on-disk file. Shape is already (1,D,H,W).
        x = torch.from_numpy(np.ascontiguousarray(self.volumes[idx]))  # uint8 (1,D,H,W)
        y = torch.tensor(int(self.labels[idx])).long()
        return x, y


def build_loaders():
    subset_dir = ensure_extracted()
    with open(os.path.join(subset_dir, "manifest.json")) as fh:
        manifest = json.load(fh)
    print("[data] manifest:", {s: manifest["splits"][s]["built_n"] for s in SPLITS})

    workers = 0 if CACHE_IN_RAM else NUM_WORKERS
    kw = dict(batch_size=BATCH_SIZE, num_workers=workers,
              pin_memory=(device == "cuda"))
    if workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)

    train_ds = OCTMemmapDataset(subset_dir, "Training",   cache_in_ram=CACHE_IN_RAM)
    val_ds   = OCTMemmapDataset(subset_dir, "Validation", cache_in_ram=CACHE_IN_RAM)
    test_ds  = OCTMemmapDataset(subset_dir, "Test",       cache_in_ram=CACHE_IN_RAM)

    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, shuffle=True,  generator=g, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
    return train_loader, val_loader, test_loader

In [ ]:
train_loader, val_loader, test_loader = build_loaders()

[data] copying archive Drive->local SSD ...
[data] copied in 30.2s
[data] extracting ...
[data] extracted in 22.7s
[data] manifest: {'Training': 2100, 'Validation': 300, 'Test': 900}


In [ ]:
# ====== cấu hình quét (A100 -> full họ) ======
SWEEP = {
    "ResNet":       ["resnet18", "resnet34", "resnet50", "resnet101", "resnet152", "resnet200"],
    "DenseNet":     ["densenet121", "densenet169", "densenet201", "densenet264"],
    "EfficientNet": ["efficientnet-b0", "efficientnet-b1", "efficientnet-b2", "efficientnet-b3",
                     "efficientnet-b4", "efficientnet-b5", "efficientnet-b6", "efficientnet-b7"],
}
SWEEP_EPOCHS = 30
LR  = 1e-4 * (BATCH_SIZE / 4) ** 0.5    # scale LR theo √batch
WD, PATIENCE = 1e-4, 10


def build_model(name, in_ch=1, n_cls=2):
    if name.startswith("resnet"):
        return getattr(nets, name)(spatial_dims=3, n_input_channels=in_ch, num_classes=n_cls)
    if name.startswith("densenet"):
        cls = {"densenet121": nets.DenseNet121, "densenet169": nets.DenseNet169,
               "densenet201": nets.DenseNet201, "densenet264": nets.DenseNet264}[name]
        return cls(spatial_dims=3, in_channels=in_ch, out_channels=n_cls)
    if name.startswith("efficientnet-"):
        return nets.EfficientNetBN(name, spatial_dims=3, in_channels=in_ch, num_classes=n_cls)
    raise ValueError(name)


@torch.no_grad()
def eval_acc(model, loader):
    model.eval(); c = t = 0
    for x, y in loader:
        x, y = to_device_normalize(x, y)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
        c += (logits.argmax(1) == y).sum().item(); t += y.numel()
    return c / t


def static_probe(name):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    try:
        m = build_model(name).to(device); opt = torch.optim.AdamW(m.parameters(), LR, weight_decay=WD)
        xb, yb = next(iter(train_loader)); xb, yb = to_device_normalize(xb, yb)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            loss = nn.functional.cross_entropy(m(xb), yb)
        loss.backward(); opt.step(); torch.cuda.synchronize()
        params = sum(p.numel() for p in m.parameters()) / 1e6
        vram = torch.cuda.max_memory_allocated() / 1e9
        del m, opt, xb, yb, loss; torch.cuda.empty_cache()
        return params, vram, "ok"
    except RuntimeError as e:
        torch.cuda.empty_cache()
        return float("nan"), float("nan"), ("OOM" if "out of memory" in str(e).lower() else "ERR")


def train_one(name):
    torch.manual_seed(SEED); torch.cuda.empty_cache()
    try:
        model = build_model(name).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=SWEEP_EPOCHS)
        scaler = torch.amp.GradScaler("cuda",
                    enabled=(USE_AMP and device == "cuda" and amp_dtype == torch.float16))
        crit = nn.CrossEntropyLoss()
        best_val = best_test = 0.0; best_ep = -1; bad = 0; ep_times = []
        for ep in range(SWEEP_EPOCHS):
            model.train(); t0 = time.time()
            for x, y in train_loader:
                x, y = to_device_normalize(x, y)
                opt.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
                    loss = crit(model(x), y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            sched.step(); ep_times.append(time.time() - t0)
            va = eval_acc(model, val_loader)
            if va > best_val:
                best_val, best_ep = va, ep; best_test = eval_acc(model, test_loader); bad = 0
            else:
                bad += 1
                if bad >= PATIENCE: break
            print(f"  [{name}] ep{ep+1:02d} val={va:.4f} (best {best_val:.4f})", end="\r")
        del model, opt; torch.cuda.empty_cache()
        return dict(val=best_val, test=best_test, best_ep=best_ep + 1,
                    sec_ep=sum(ep_times) / len(ep_times), status="ok")
    except RuntimeError as e:
        torch.cuda.empty_cache()
        return dict(val=float("nan"), test=float("nan"), best_ep=-1, sec_ep=float("nan"),
                    status="OOM" if "out of memory" in str(e).lower() else "ERR")


# ====== chạy ======
rows = []
for family, names in SWEEP.items():
    for name in names:
        print(f"\n=== [{family}] {name} ===")
        params, vram, st = static_probe(name)
        if st != "ok":
            print(f"  static probe: {st}")
            rows.append(dict(family=family, name=name, params=params, vram=vram,
                             val=float("nan"), test=float("nan"), best_ep=-1, sec_ep=float("nan"), status=st))
            continue
        print(f"  params {params:.1f}M | VRAM {vram:.1f}GB")
        r = train_one(name); r.update(family=family, name=name, params=params, vram=vram)
        rows.append(r)
        print(f"  val={r['val']:.4f} test={r['test']:.4f} @ep{r['best_ep']} | {r['sec_ep']:.1f}s/ep   ")

# ====== bảng ======
def fmt(r):
    return (f"{r['name']:18s} {r['params']:>6.1f}M {r['vram']:>5.1f}G {r['sec_ep']:>5.1f}s "
            f"{r['val']:>7.4f} {r['test']:>7.4f} {r['best_ep']:>6d}  {r['status']}")

hdr = f"{'model':18s} {'params':>7} {'VRAM':>6} {'s/ep':>6} {'Val':>7} {'Test':>7} {'bestEp':>6}"
print("\n================= SWEEP 3 HỌ (có augmentation) =================")
for family in SWEEP:
    print(f"\n-- {family} --\n{hdr}")
    for r in rows:
        if r["family"] == family: print(fmt(r))

ok = sorted([r for r in rows if r["status"] == "ok"], key=lambda r: -r["val"])
print("\n----- TOP theo Val -----\n" + hdr)
for r in ok[:5]: print(fmt(r))
if ok:
    w = ok[0]
    print(f"\n>> WINNER: {w['name']} ({w['family']}) Val={w['val']:.4f} Test={w['test']:.4f} "
          f"| {w['params']:.1f}M | {w['vram']:.1f}GB")

# lưu kết quả ra Drive (tùy chọn)
try:
    with open("/content/drive/MyDrive/MasterBKDN/Thesis/sweep_results.json", "w") as fh:
        json.dump(rows, fh, indent=2)
    print("\n[saved] sweep_results.json -> Drive")
except Exception as e:
    print("save skipped:", e)



=== [ResNet] resnet18 ===
  params 33.2M | VRAM 8.7GB
  val=0.7900 test=0.7722 @ep10 | 41.8s/ep   

=== [ResNet] resnet34 ===
  params 63.5M | VRAM 10.6GB
  val=0.7933 test=0.7667 @ep10 | 50.5s/ep   

=== [ResNet] resnet50 ===
  params 46.2M | VRAM 21.9GB
  val=0.7567 test=0.7189 @ep14 | 64.2s/ep   

=== [ResNet] resnet101 ===
  params 85.2M | VRAM 25.5GB
  val=0.7833 test=0.7411 @ep14 | 72.5s/ep   

=== [ResNet] resnet152 ===
  params 117.4M | VRAM 31.2GB
  val=0.7833 test=0.7567 @ep5 | 84.9s/ep   

=== [ResNet] resnet200 ===
  static probe: OOM

=== [DenseNet] densenet121 ===
  params 11.2M | VRAM 3.9GB
  val=0.7800 test=0.7667 @ep4 | 11.3s/ep   

=== [DenseNet] densenet169 ===
  params 18.5M | VRAM 4.1GB
  val=0.8100 test=0.7467 @ep12 | 14.1s/ep   

=== [DenseNet] densenet201 ===
  params 25.3M | VRAM 4.7GB
  val=0.7967 test=0.7533 @ep11 | 16.2s/ep   

=== [DenseNet] densenet264 ===
  params 40.3M | VRAM 5.4GB
  val=0.8033 test=0.7356 @ep27 | 20.1s/ep   

=== [EfficientNet] efficie

In [ ]:
# Đặt CUỐI tác vụ. Sau khi xong, runtime tự kill -> giải phóng GPU ngay.
from google.colab import runtime
runtime.unassign()
